# 위성영상 딥러닝 핸즈온: 산불 탐지 & 식생 건강 변화 분석
## 산림자원학과 실습 | Google Colab

**사례 지역**: 2023년 강릉·동해 산불 (2023-04-11 발화)  
**위성**: Copernicus Sentinel-2 (ESA, 무료)  
**플랫폼**: Google Earth Engine + PyTorch + scikit-learn

---

### 예상 실습 시간: 약 90분

| 단계 | 내용 | 예상 시간 |
|------|------|-----------|
| 1–2 | 환경 설정 & GEE 인증 | 10분 |
| 3–4 | 위성영상 로드 & 시각화 | 20분 |
| 5 | 식생건강지수 계산 (NDVI·NBR·dNBR) | 20분 |
| 6 | 딥러닝 산불 탐지 파이프라인 | 25분 |
| 7–8 | 비교 분석 & 통계 | 15분 |

## 학습 목표

1. **Sentinel-2** 위성영상의 밴드 구성을 이해하고 진색·위색합성을 수행할 수 있다
2. **NDVI, NBR, dNBR** 등 식생건강지수를 직접 계산하고 해석할 수 있다
3. **Google Earth Engine(GEE)** 플랫폼에서 대용량 위성 데이터를 처리하는 방법을 익힌다
4. **딥러닝 기반 분류기(MLP)**로 산불 피해 영역을 자동 탐지하는 AI 파이프라인을 구현한다
5. 이전·이후 영상 비교를 통해 산림 피해를 **정량적으로 분석**할 수 있다

---

## 핵심 개념 정리

| 용어 | 설명 |
|------|---------|
| **Sentinel-2** | ESA(유럽우주국) 운영, 10~60m 해상도, 13개 밴드 광학 위성 |
| **NDVI** | 정규화 식생 지수 (NIR-Red)/(NIR+Red), 식생 활력 측정 (-1~1) |
| **NBR** | 정규화 연소 비율 (NIR-SWIR2)/(NIR+SWIR2), 산불 피해에 특화 |
| **dNBR** | 산불 이전 NBR - 이후 NBR, 피해 심각도 정량화 (양수=피해) |
| **U-Net** | 위성영상 분할(Segmentation)에 많이 쓰이는 CNN 인코더-디코더 |
| **GEE** | Google Earth Engine, 페타바이트급 위성 데이터 클라우드 플랫폼 |

---

## Sentinel-2 주요 밴드

| 밴드 | 이름 | 파장(nm) | 해상도(m) | 주요 용도 |
|------|------|----------|-----------|----------|
| B2 | Blue | 490 | 10 | 수계, 대기 보정 |
| B3 | Green | 560 | 10 | 식생 활력 |
| B4 | Red | 665 | 10 | 엽록소 흡수 |
| B8 | NIR | 842 | 10 | 식생 생체량 |
| B11 | SWIR1 | 1610 | 20 | 수분 함량 |
| B12 | SWIR2 | 2190 | 20 | **연소 탐지 핵심** |

> **NBR 산불 탐지 원리**: 건강한 식생 = NIR 높음 + SWIR 낮음 → NBR 양수  
> 산불 피해 지역 = NIR 낮음 + SWIR 높음 → NBR 음수 → **dNBR 큰 양수값**

---
## 섹션 1. 환경 설정

In [ ]:
# 필요 패키지 설치 (Colab에 없는 것만)
!pip install -q geemap segmentation-models-pytorch==1.3.1
print("패키지 설치 완료")

In [ ]:
import ee
import geemap
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import BoundaryNorm, ListedColormap
import requests
from PIL import Image as PILImage
import io
import os
import warnings

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import segmentation_models_pytorch as smp
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

warnings.filterwarnings('ignore')
plt.rcParams['figure.dpi'] = 100

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"PyTorch 버전: {torch.__version__}")
print(f"사용 장치: {device}")
print(f"GPU 사용 가능: {torch.cuda.is_available()}")

---
## 섹션 2. Google Earth Engine 인증

### GEE 계정 준비 방법
1. Google 계정으로 https://earthengine.google.com/ 접속
2. **"Get Started"** 클릭 → 학술 용도(Noncommercial)로 무료 등록
3. 승인 후(즉시~1일 소요) 아래 셀 실행

> **처음 사용**: 셀 실행 후 출력되는 인증 링크를 클릭 → Google 로그인 → 코드 복사·붙여넣기

In [ ]:
# GEE 인증 및 초기화
try:
    geemap.ee_initialize()
    print("GEE 인증 성공!")
except Exception as e:
    print(f"자동 인증 실패: {e}")
    print("아래 셀의 수동 인증을 시도하세요.")

In [ ]:
# 자동 인증 실패 시 수동 인증 (주석 해제 후 실행)
# ee.Authenticate()
# ee.Initialize(project='your-gee-project-id')  # GEE 프로젝트 ID 입력

# 연결 확인
try:
    test = ee.Number(42).getInfo()
    print(f"GEE 연결 확인: {test} (정상)")
except Exception as e:
    print(f"GEE 연결 실패: {e}")

---
## 섹션 3. 연구 지역 설정: 2023 강릉·동해 산불

### 사건 개요
| 항목 | 내용 |
|------|---------|
| **발화 일시** | 2023년 4월 11일 (화요일) |
| **발화 지점** | 강원도 강릉시 옥계면 일대 |
| **피해 규모** | 약 2,148 ha (임야 및 주거지) |
| **피해 지역** | 강릉시 · 동해시 일대 |
| **원인** | 강풍을 동반한 건조한 날씨 |

**분석 기간 설정 전략**
- 이전 영상: 2023년 2~3월 (산불 전 식생 기준값)
- 이후 영상: 2023년 4월 하순~5월 (산불 직후 피해 상태)

In [ ]:
# ============================================================
# 연구 지역 및 날짜 설정
# ============================================================

# 전체 분석 영역 (ROI: Region of Interest)
ROI = ee.Geometry.Rectangle([128.85, 37.55, 129.25, 37.95])

# 딥러닝 분석용 소영역 (다운로드 속도 최적화)
ROI_DL = ee.Geometry.Rectangle([128.95, 37.62, 129.20, 37.90])

# 날짜 설정 (산불: 2023-04-11)
PRE_START  = '2023-02-01'
PRE_END    = '2023-03-31'
POST_START = '2023-04-20'
POST_END   = '2023-05-31'

# 분석 밴드 (Blue, Green, Red, NIR, SWIR1, SWIR2)
BANDS = ['B2', 'B3', 'B4', 'B8', 'B11', 'B12']

# 지도 중심 좌표
CENTER = [37.73, 129.05]

# ============================================================
# 연구 지역 인터랙티브 지도 표시
# ============================================================
Map = geemap.Map(center=CENTER, zoom=10)
Map.add_layer(ROI,    {'color': 'blue'},  '전체 분석 영역')
Map.add_layer(ROI_DL, {'color': 'red'},   'DL 분석 영역 (소영역)')
Map.add_layer(
    ee.Geometry.Point([129.05, 37.73]),
    {'color': 'orange', 'pointRadius': 10},
    '강릉 옥계 (산불 발화점 부근)'
)
print("연구 지역 설정 완료")
print(f"이전 영상: {PRE_START} ~ {PRE_END}")
print(f"이후 영상: {POST_START} ~ {POST_END}")
Map

---
## 섹션 4. Sentinel-2 위성영상 로드

`COPERNICUS/S2_SR_HARMONIZED` 컬렉션 사용
- **Level-2A**: 대기 보정 완료 (Surface Reflectance)
- **값 범위**: 0~10000 (반사율 × 10000)
- 여러 날짜를 **median 합성**하여 구름 영향 제거

In [ ]:
def load_sentinel2_median(start_date, end_date, roi, max_cloud_pct=20):
    """
    Sentinel-2 SR 영상 로드 및 median 합성

    Parameters
    ----------
    start_date : str  시작 날짜 'YYYY-MM-DD'
    end_date   : str  종료 날짜 'YYYY-MM-DD'
    roi        : ee.Geometry  관심 영역
    max_cloud_pct : int  최대 허용 운량(%)

    Returns
    -------
    ee.Image  클리핑된 median 합성 영상
    """
    collection = (
        ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
        .filterBounds(roi)
        .filterDate(start_date, end_date)
        .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', max_cloud_pct))
    )
    count = collection.size().getInfo()
    print(f"  사용 가능 영상: {count}장")

    # median 합성: 각 픽셀에서 시간축 중앙값 → 구름 자동 제거
    return collection.median().clip(roi), count


print("[이전 영상] 로드 중...")
pre_img, pre_cnt = load_sentinel2_median(PRE_START, PRE_END, ROI)

print("\n[이후 영상] 로드 중...")
post_img, post_cnt = load_sentinel2_median(POST_START, POST_END, ROI)

print(f"\n영상 로드 완료 — 이전 {pre_cnt}장 / 이후 {post_cnt}장 합성")

In [ ]:
# ============================================================
# GEE 인터랙티브 지도: 이전/이후 진색·위색합성 비교
# ============================================================

Map2 = geemap.Map(center=CENTER, zoom=11)

# 시각화 파라미터
vis_rgb  = {'bands': ['B4', 'B3', 'B2'], 'min': 0, 'max': 3000, 'gamma': 1.4}
vis_nrg  = {'bands': ['B8', 'B4', 'B3'], 'min': 0, 'max': 4000, 'gamma': 1.4}
vis_swir = {'bands': ['B12', 'B8', 'B3'], 'min': 0, 'max': 3000, 'gamma': 1.4}

Map2.add_layer(pre_img,  vis_rgb,  '이전 진색합성 (RGB)')
Map2.add_layer(post_img, vis_rgb,  '이후 진색합성 (RGB)')
Map2.add_layer(pre_img,  vis_nrg,  '이전 위색합성 (NRG)')
Map2.add_layer(post_img, vis_nrg,  '이후 위색합성 (NRG)')
Map2.add_layer(post_img, vis_swir, '이후 SWIR합성 (산불 탐지용)')

print("레이어 토글: 지도 우측 상단 레이어 아이콘을 클릭하세요")
print("위색합성 해석: 식생=붉은색, 산불피해=회색/갈색, 도시=청회색")
Map2

In [ ]:
# ============================================================
# 정적 이미지 시각화 (matplotlib)
# ============================================================

def get_thumb_image(image, bands, min_val, max_val, gamma=1.4, size=512):
    """GEE 이미지를 NumPy 배열로 변환 (썸네일 URL 이용)"""
    url = image.getThumbURL({
        'dimensions': size,
        'region': ROI,
        'bands': bands,
        'min': min_val,
        'max': max_val,
        'gamma': gamma,
        'format': 'png'
    })
    resp = requests.get(url, timeout=90)
    return np.array(PILImage.open(io.BytesIO(resp.content)))


print("이미지 다운로드 중 (약 30초)...")
img_pre_rgb  = get_thumb_image(pre_img,  ['B4','B3','B2'], 0, 3000)
img_post_rgb = get_thumb_image(post_img, ['B4','B3','B2'], 0, 3000)
img_pre_nrg  = get_thumb_image(pre_img,  ['B8','B4','B3'], 0, 4000)
img_post_nrg = get_thumb_image(post_img, ['B8','B4','B3'], 0, 4000)
print("다운로드 완료")

fig, axes = plt.subplots(2, 2, figsize=(13, 11))
fig.suptitle('Sentinel-2 위성영상: 2023 강릉·동해 산불 이전/이후 비교',
             fontsize=13, fontweight='bold')

titles = [
    '이전 진색합성 (2023-02~03)',
    '이후 진색합성 (2023-04~05)',
    '이전 위색합성 NIR-Red-Green',
    '이후 위색합성 NIR-Red-Green'
]
imgs = [img_pre_rgb, img_post_rgb, img_pre_nrg, img_post_nrg]

for ax, img, title in zip(axes.flat, imgs, titles):
    ax.imshow(img)
    ax.set_title(title, fontsize=11)
    ax.axis('off')

fig.text(0.5, 0.01,
         '위색합성: 건강한 식생=빨간색  |  산불 피해=회색/갈색  |  도시=청회색',
         ha='center', fontsize=10, style='italic', color='#444')
plt.tight_layout()
plt.savefig('satellite_comparison.png', bbox_inches='tight', dpi=150)
plt.show()
print("'satellite_comparison.png' 저장 완료")

---
## 섹션 5. 식생건강지수 계산 및 시각화

### 지수 공식

$$NDVI = \frac{NIR - Red}{NIR + Red} = \frac{B8 - B4}{B8 + B4}$$

$$NBR = \frac{NIR - SWIR2}{NIR + SWIR2} = \frac{B8 - B12}{B8 + B12}$$

$$dNBR = NBR_{\text{이전}} - NBR_{\text{이후}}$$

### dNBR 심각도 분류 기준 (UN-SPIDER)

| dNBR 범위 | 피해 등급 |
|-----------|----------|
| < −0.1 | 식생 증가 (재생) |
| −0.1 ~ 0.1 | 피해 없음 |
| 0.1 ~ 0.27 | 경미 (Low) |
| 0.27 ~ 0.44 | 중간 (Moderate-Low) |
| 0.44 ~ 0.66 | 높음 (Moderate-High) |
| > 0.66 | 극심 (High) |

In [ ]:
# ============================================================
# GEE에서 식생건강지수 계산
# ============================================================

# NDVI: 식생 활력 지수
ndvi_pre  = pre_img.normalizedDifference(['B8', 'B4']).rename('NDVI')
ndvi_post = post_img.normalizedDifference(['B8', 'B4']).rename('NDVI')
ndvi_diff = ndvi_post.subtract(ndvi_pre).rename('dNDVI')

# NBR: 산불 특화 지수 (Normalized Burn Ratio)
nbr_pre  = pre_img.normalizedDifference(['B8', 'B12']).rename('NBR')
nbr_post = post_img.normalizedDifference(['B8', 'B12']).rename('NBR')

# dNBR: 산불 심각도 (이전 - 이후, 양수 = 피해)
dnbr = nbr_pre.subtract(nbr_post).rename('dNBR')

print("식생건강지수 계산 완료")
print("  - NDVI 이전 / 이후 / 변화량 (dNDVI)")
print("  - NBR 이전 / 이후")
print("  - dNBR (연소 심각도 지표)")

In [ ]:
# ============================================================
# 지수 시각화 (6-panel)
# ============================================================

def get_index_image(image, min_val, max_val, palette, size=512):
    """GEE 단일 밴드 이미지를 컬러맵으로 다운로드"""
    url = image.getThumbURL({
        'dimensions': size,
        'region': ROI,
        'min': min_val,
        'max': max_val,
        'palette': palette,
        'format': 'png'
    })
    resp = requests.get(url, timeout=90)
    return np.array(PILImage.open(io.BytesIO(resp.content)))


# 팔레트 정의
NDVI_PAL  = ['d73027','f46d43','fdae61','fee08b','d9ef8b','a6d96a','1a9850']
NBR_PAL   = ['d73027','fc8d59','fee08b','d9ef8b','91cf60','1a9850']
DNBR_PAL  = ['1a9850','66bd63','d9ef8b','fee08b','fdae61','f46d43','d73027']
DIFF_PAL  = ['d73027','fc8d59','ffffbf','91cf60','1a9850']

print("지수 이미지 다운로드 중 (약 60초)...")
idx_imgs = {
    'NDVI 이전 (2023-02~03)':     get_index_image(ndvi_pre,  -0.1, 0.9,  NDVI_PAL),
    'NDVI 이후 (2023-04~05)':     get_index_image(ndvi_post, -0.1, 0.9,  NDVI_PAL),
    'NDVI 변화 (이후-이전)':       get_index_image(ndvi_diff, -0.6, 0.1,  DIFF_PAL),
    'NBR 이전':                   get_index_image(nbr_pre,   -0.1, 0.8,  NBR_PAL),
    'NBR 이후':                   get_index_image(nbr_post,  -0.1, 0.8,  NBR_PAL),
    'dNBR (피해 심각도)':          get_index_image(dnbr,      -0.2, 0.8,  DNBR_PAL),
}
print("다운로드 완료")

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.suptitle('식생건강지수 분석: NDVI · NBR · dNBR\n2023 강릉·동해 산불',
             fontsize=13, fontweight='bold')

for ax, (title, img) in zip(axes.flat, idx_imgs.items()):
    ax.imshow(img)
    ax.set_title(title, fontsize=10)
    ax.axis('off')

plt.tight_layout()
plt.savefig('vegetation_indices.png', bbox_inches='tight', dpi=150)
plt.show()
print("'vegetation_indices.png' 저장 완료")

In [ ]:
# ============================================================
# dNBR 심각도 분류 및 면적 계산
# ============================================================

# UN-SPIDER 기준 임계값
DNBR_BINS   = [-2.0, -0.1, 0.10, 0.27, 0.44, 0.66, 3.0]
SEV_LABELS  = ['식생 증가', '피해 없음', '경미(Low)',
               '중간(Mod-Low)', '높음(Mod-High)', '극심(High)']
SEV_COLORS  = ['#2166ac', '#d9f0a3', '#fee08b', '#fc8d59', '#e31a1c', '#800026']

# GEE에서 분류 이미지 생성
classes = []
for i in range(len(DNBR_BINS) - 1):
    lo, hi = DNBR_BINS[i], DNBR_BINS[i + 1]
    cls = dnbr.gte(lo).And(dnbr.lt(hi)).multiply(i + 1)
    classes.append(cls)

# 각 클래스 이미지를 합산
severity_img = ee.ImageCollection(classes).sum().rename('severity')

# 등급별 면적 계산 (30m² → ha)
print("dNBR 심각도 면적 계산 중 (약 30초)...")
areas = {}
for i, label in enumerate(SEV_LABELS):
    result = (
        severity_img.eq(i + 1)
        .multiply(ee.Image.pixelArea())
        .reduceRegion(
            reducer=ee.Reducer.sum(),
            geometry=ROI,
            scale=20,
            maxPixels=1e9
        ).getInfo()
    )
    areas[label] = result.get('severity', 0) / 10000.0
    print(f"  {label}: {areas[label]:.1f} ha")

print(f"\n총 분석 면적: {sum(areas.values()):.0f} ha")

In [ ]:
# 심각도 지도 시각화
SEV_PAL_GEE = [c.lstrip('#') for c in SEV_COLORS]
sev_img_arr = get_index_image(severity_img, 1, 6, SEV_PAL_GEE)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('dNBR 기반 산불 피해 심각도 분류\n2023 강릉·동해 산불',
             fontsize=13, fontweight='bold')

# 심각도 지도
axes[0].imshow(sev_img_arr)
axes[0].set_title('피해 심각도 분포도', fontsize=11)
axes[0].axis('off')
legend_patches = [
    mpatches.Patch(color=c, label=f"{l}: {areas.get(l,0):.0f} ha")
    for c, l in zip(SEV_COLORS, SEV_LABELS)
]
axes[0].legend(handles=legend_patches, loc='lower left', fontsize=9, framealpha=0.9)

# 면적 막대그래프
labels_plot = [l for l in SEV_LABELS if areas.get(l, 0) > 1]
values_plot = [areas[l] for l in labels_plot]
colors_plot = [SEV_COLORS[SEV_LABELS.index(l)] for l in labels_plot]

bars = axes[1].barh(labels_plot, values_plot, color=colors_plot,
                    edgecolor='white', linewidth=0.5)
axes[1].set_xlabel('면적 (ha)', fontsize=11)
axes[1].set_title('피해 등급별 면적', fontsize=11)
for bar, v in zip(bars, values_plot):
    axes[1].text(v + 5, bar.get_y() + bar.get_height()/2,
                 f'{v:.0f} ha', va='center', fontsize=9)
axes[1].grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.savefig('dnbr_severity_map.png', bbox_inches='tight', dpi=150)
plt.show()
print("'dnbr_severity_map.png' 저장 완료")

---
## 섹션 6. 딥러닝 기반 산불 탐지

### 접근 방법 개요

```
[Sentinel-2 이후 영상]
   6개 밴드 픽셀값 + 스펙트럴 지수 (NDVI, NBR, ...)
              ↓
   [딥러닝 분류기 (MLP)]
              ↓
   산불 피해 확률 맵 (0~1)
```

### 의사 레이블(Pseudo-Label) 전략
현장 조사 없이 AI 모델을 학습시키는 방법:
1. 물리 기반 지수 **(dNBR)**으로 신뢰도 높은 픽셀 선별
2. 선별 픽셀을 **학습 레이블**로 사용
3. 딥러닝 모델이 **스펙트럴 패턴** 학습
4. 전체 영역에 **추론(Inference)** 적용

> 이 방법은 실무에서 레이블 데이터 부족 시 사용하는 **반지도학습(Semi-supervised)** 기법입니다.

In [ ]:
# ============================================================
# 딥러닝 분석용 영상 다운로드 (GeoTIFF)
# ============================================================
import rasterio

os.makedirs('/content/data', exist_ok=True)
PRE_FILE  = '/content/data/pre_fire.tif'
POST_FILE = '/content/data/post_fire.tif'
DNBR_FILE = '/content/data/dnbr.tif'

def download_if_missing(image, filepath, region, scale=30, label=''):
    if os.path.exists(filepath):
        print(f"  이미 존재: {filepath}")
    else:
        print(f"  다운로드 중: {label}...")
        geemap.download_ee_image(
            image=image,
            filename=filepath,
            region=region,
            scale=scale,
            crs='EPSG:4326'
        )
        print(f"  저장 완료: {filepath}")

download_if_missing(pre_img.select(BANDS).clip(ROI_DL),
                    PRE_FILE,  ROI_DL, 30, '이전 영상 (6밴드)')
download_if_missing(post_img.select(BANDS).clip(ROI_DL),
                    POST_FILE, ROI_DL, 30, '이후 영상 (6밴드)')
download_if_missing(dnbr.clip(ROI_DL),
                    DNBR_FILE, ROI_DL, 30, 'dNBR')

print("\n모든 파일 준비 완료")

In [ ]:
# ============================================================
# GeoTIFF → NumPy 배열 변환
# ============================================================

def load_tif(filepath):
    """GeoTIFF를 (bands, H, W) float32 배열로 로드"""
    with rasterio.open(filepath) as src:
        data = src.read().astype(np.float32)
        meta = src.meta
    return data, meta

pre_data,  pre_meta  = load_tif(PRE_FILE)
post_data, post_meta = load_tif(POST_FILE)
dnbr_data, _         = load_tif(DNBR_FILE)

# 유효 픽셀 마스크 (nodata=0 제거)
valid = (pre_data[0] > 0) & (post_data[0] > 0) & np.isfinite(dnbr_data[0])

H, W = pre_data.shape[1], pre_data.shape[2]
print(f"영상 크기: {H} × {W} 픽셀  ({H*W:,}개)")
print(f"유효 픽셀: {valid.sum():,}개 ({valid.mean()*100:.1f}%)")
print(f"밴드 수: {pre_data.shape[0]} ({', '.join(BANDS)})")
print(f"\n반사율 예시 (B4/Red, 이전):")
print(f"  최솟값={pre_data[2][valid].min():.0f}  최댓값={pre_data[2][valid].max():.0f}")
print(f"  평균={pre_data[2][valid].mean():.0f}  → ÷10000 = {pre_data[2][valid].mean()/10000:.4f} 반사율")

In [ ]:
# ============================================================
# 피처 엔지니어링: 스펙트럴 지수 계산 (NumPy)
# ============================================================

EPS = 1e-10  # 0 나눗셈 방지

def compute_spectral_indices(data, scale=10000.0):
    """
    Sentinel-2 배열에서 스펙트럴 지수 계산
    밴드 순서: [B2, B3, B4, B8, B11, B12]
    """
    b2, b3, b4, b8, b11, b12 = [data[i] / scale for i in range(6)]

    ndvi = (b8 - b4)  / (b8 + b4  + EPS)   # 식생 활력
    nbr  = (b8 - b12) / (b8 + b12 + EPS)   # 연소 비율
    ndwi = (b3 - b8)  / (b3 + b8  + EPS)   # 수분 지수
    evi  = 2.5 * (b8 - b4) / (b8 + 6*b4 - 7.5*b2 + 1 + EPS)  # 향상 식생

    return np.stack([ndvi, nbr, ndwi, evi], axis=0)  # (4, H, W)

IDX_NAMES = ['NDVI', 'NBR', 'NDWI', 'EVI']

pre_idx  = compute_spectral_indices(pre_data)
post_idx = compute_spectral_indices(post_data)

print("스펙트럴 지수 계산 완료")
print(f"피처 구성: {BANDS + IDX_NAMES}  →  총 {len(BANDS)+len(IDX_NAMES)}개 피처/픽셀")
print(f"\n이후 영상 NDVI 통계:")
print(f"  평균={post_idx[0][valid].mean():.3f}  표준편차={post_idx[0][valid].std():.3f}")

In [ ]:
# 피처 탐색 시각화 (이후 영상 기준)
fig, axes = plt.subplots(2, 5, figsize=(17, 7))
fig.suptitle('이후 영상 피처 맵 (밴드 + 스펙트럴 지수)', fontsize=12)

all_features_vis = np.vstack([post_data / 10000.0, post_idx])  # (10, H, W)
feature_names = BANDS + IDX_NAMES
band_ranges = [
    (0, 0.2), (0, 0.2), (0, 0.2), (0, 0.4), (0, 0.3), (0, 0.3),
    (-0.3, 0.8), (-0.5, 0.7), (-0.5, 0.5), (-0.3, 0.6)
]

for ax, feat, name, (vmin, vmax) in zip(
        axes.flat, all_features_vis, feature_names, band_ranges):
    feat_vis = feat.copy()
    feat_vis[~valid] = np.nan
    im = ax.imshow(feat_vis, cmap='RdYlGn', vmin=vmin, vmax=vmax)
    ax.set_title(name, fontsize=9)
    ax.axis('off')
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.02)

plt.tight_layout()
plt.savefig('feature_maps.png', bbox_inches='tight', dpi=150)
plt.show()

In [ ]:
# ============================================================
# 의사 레이블 생성 (Pseudo-Labeling from dNBR)
# ============================================================

BURN_THRESH   = 0.27   # 이 값 이상: 피해 픽셀
NORMAL_THRESH = 0.10   # 이 값 이하: 정상 픽셀

dnbr_flat  = dnbr_data[0].flatten()
valid_flat = valid.flatten()

# 확실한 피해 / 확실한 정상 픽셀 마스크
burn_flat   = (dnbr_flat > BURN_THRESH)   & valid_flat
normal_flat = (dnbr_flat < NORMAL_THRESH) & (dnbr_flat > -0.1) & valid_flat

n_burn   = burn_flat.sum()
n_normal = normal_flat.sum()
n_sample = min(n_burn, n_normal, 60000)  # 클래스 균형 맞추기

print(f"의사 레이블 통계")
print(f"  피해 픽셀 (dNBR>{BURN_THRESH}): {n_burn:,}개")
print(f"  정상 픽셀 (dNBR<{NORMAL_THRESH}): {n_normal:,}개")
print(f"  학습용 샘플 (균형): {n_sample:,}개 × 2클래스")
print(f"  불확실 구간 ({NORMAL_THRESH}~{BURN_THRESH}): 학습 제외")

# 전체 피처 행렬 구성 (H*W, 10)
all_feats = np.hstack([
    (post_data / 10000.0).reshape(6, -1).T,
    post_idx.reshape(4, -1).T
]).astype(np.float32)

# 서브샘플링 및 레이블 생성
np.random.seed(42)
burn_idx   = np.random.choice(np.where(burn_flat)[0],   n_sample, replace=False)
normal_idx = np.random.choice(np.where(normal_flat)[0], n_sample, replace=False)

X_train = np.vstack([all_feats[burn_idx], all_feats[normal_idx]])
y_train = np.hstack([np.ones(n_sample),   np.zeros(n_sample)])

# 셔플
shuf = np.random.permutation(len(X_train))
X_train, y_train = X_train[shuf], y_train[shuf]

print(f"\n학습 데이터 준비 완료: {X_train.shape}")

### 딥러닝 모델: 픽셀 기반 MLP (Multi-Layer Perceptron)

```
입력층  (10 피처: 6밴드 + NDVI/NBR/NDWI/EVI)
   ↓  Linear(10→64) + BatchNorm + ReLU + Dropout(0.3)
은닉층1 (64 뉴런)
   ↓  Linear(64→32) + BatchNorm + ReLU + Dropout(0.2)
은닉층2 (32 뉴런)
   ↓  Linear(32→16) + ReLU
은닉층3 (16 뉴런)
   ↓  Linear(16→1) + Sigmoid
출력층  (산불 피해 확률: 0~1)
```

각 픽셀을 독립적으로 분류하는 **픽셀 기반(pixel-wise)** 분류 방식.

In [ ]:
# ============================================================
# PyTorch MLP 모델 정의
# ============================================================

class BurnAreaMLP(nn.Module):
    """
    산불 피해 영역 탐지 MLP
    입력: Sentinel-2 6밴드 + 4개 스펙트럴 지수 = 10 피처
    출력: 산불 피해 확률 (0~1)
    """
    def __init__(self, in_features: int = 10):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_features, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(64, 32),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.Dropout(0.2),

            nn.Linear(32, 16),
            nn.ReLU(),

            nn.Linear(16, 1),
            nn.Sigmoid(),
        )

    def forward(self, x):
        return self.net(x).squeeze(-1)


mlp = BurnAreaMLP(in_features=10).to(device)
total_params = sum(p.numel() for p in mlp.parameters())
print(f"모델 아키텍처:\n{mlp}")
print(f"\n총 파라미터: {total_params:,}개")

In [ ]:
# ============================================================
# MLP 학습
# ============================================================

# 텐서 변환 및 데이터로더
X_t = torch.FloatTensor(X_train).to(device)
y_t = torch.FloatTensor(y_train).to(device)
dataset = TensorDataset(X_t, y_t)
loader  = DataLoader(dataset, batch_size=4096, shuffle=True)

# 손실 함수, 옵티마이저
criterion = nn.BCELoss()
optimizer = optim.Adam(mlp.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=40)

EPOCHS = 40
train_losses, train_accs = [], []

mlp.train()
print(f"학습 시작  |  Epochs: {EPOCHS}  |  Batch: 4096  |  Device: {device}")
print("-" * 55)

for epoch in range(EPOCHS):
    ep_loss, correct, total = 0.0, 0, 0
    for Xb, yb in loader:
        optimizer.zero_grad()
        pred = mlp(Xb)
        loss = criterion(pred, yb)
        loss.backward()
        optimizer.step()
        ep_loss += loss.item() * len(yb)
        correct += ((pred > 0.5) == yb.bool()).sum().item()
        total   += len(yb)
    scheduler.step()
    avg_loss = ep_loss / total
    acc = correct / total * 100
    train_losses.append(avg_loss)
    train_accs.append(acc)
    if (epoch + 1) % 8 == 0:
        print(f"Epoch [{epoch+1:2d}/{EPOCHS}]  Loss: {avg_loss:.4f}  Acc: {acc:.1f}%")

print("-" * 55)
print(f"최종 정확도: {train_accs[-1]:.1f}%")

In [ ]:
# 학습 곡선 시각화
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(range(1, EPOCHS+1), train_losses, 'b-', linewidth=1.5)
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss (BCE)')
ax1.set_title('학습 손실 (Training Loss)')
ax1.grid(True, alpha=0.3)

ax2.plot(range(1, EPOCHS+1), train_accs, 'g-', linewidth=1.5)
ax2.axhline(90, color='r', linestyle='--', alpha=0.5, label='90% 기준선')
ax2.set_xlabel('Epoch'); ax2.set_ylabel('Accuracy (%)')
ax2.set_title('학습 정확도 (Training Accuracy)')
ax2.set_ylim(50, 101); ax2.legend(); ax2.grid(True, alpha=0.3)

plt.suptitle('MLP 모델 학습 곡선', fontsize=13)
plt.tight_layout()
plt.savefig('training_curves.png', bbox_inches='tight', dpi=150)
plt.show()

In [ ]:
# ============================================================
# 전체 영역 추론 (Inference)
# ============================================================

mlp.eval()
BATCH_SIZE = 20000
prob_flat = np.zeros(H * W, dtype=np.float32)

with torch.no_grad():
    for i in range(0, H * W, BATCH_SIZE):
        batch = torch.FloatTensor(all_feats[i:i+BATCH_SIZE]).to(device)
        prob_flat[i:i+BATCH_SIZE] = mlp(batch).cpu().numpy()

# 2D 맵으로 복원
prob_map = prob_flat.reshape(H, W)
burn_pred = (prob_map > 0.5).astype(np.uint8)

# nodata 영역 처리
prob_map[~valid]  = np.nan
burn_pred[~valid] = 0

# 면적 계산 (30m 해상도 가정)
pixel_area_ha    = (30 * 30) / 10000.0
mlp_burned_ha    = burn_pred.sum() * pixel_area_ha
dnbr_burned_ha   = ((dnbr_data[0] > BURN_THRESH) & valid).sum() * pixel_area_ha

print(f"추론 완료")
print(f"  MLP 탐지 산불 피해 면적:   {mlp_burned_ha:.0f} ha")
print(f"  dNBR 기준 산불 피해 면적:  {dnbr_burned_ha:.0f} ha")
print(f"  두 방법 면적 차이:          {abs(mlp_burned_ha - dnbr_burned_ha):.0f} ha")

---
### 심화: U-Net 이미지 세그멘테이션 아키텍처

MLP는 각 픽셀을 독립적으로 처리하지만, **U-Net**은 주변 공간 맥락(Context)을 함께 고려합니다.

```
입력 영상 (H×W×6)
       │
  Encoder (ResNet34)
  ┌─────────────────────────────┐
  │  64 → 128 → 256 → 512 채널  │  ← 패턴 압축 (스펙트럴 특징 추출)
  └─────────────────────────────┘
       │  Skip Connections
  Decoder
  ┌─────────────────────────────┐
  │  512 → 256 → 128 → 64 채널  │  ← 공간 해상도 복원
  └─────────────────────────────┘
       │
  출력 마스크 (H×W×1, Sigmoid)
```

**Skip Connection**: Encoder 각 단계의 세밀한 공간 정보를 Decoder에 직접 전달  
→ 경계선이 정밀한 분할(Segmentation) 가능

> 실제 운용 시에는 labeled 위성영상(수천~수만 패치)으로 파인튜닝 필요

In [ ]:
# ============================================================
# U-Net 아키텍처 설정 (segmentation-models-pytorch)
# ============================================================

unet = smp.Unet(
    encoder_name='resnet34',     # ResNet34 인코더
    encoder_weights='imagenet',  # ImageNet 사전학습 가중치 로드
    in_channels=6,               # Sentinel-2 6밴드
    classes=1,                   # 이진 분류 (피해/비피해)
    activation='sigmoid',
)
unet = unet.to(device)
unet.eval()

unet_total  = sum(p.numel() for p in unet.parameters())
unet_enc    = sum(p.numel() for p in unet.encoder.parameters())
print(f"U-Net 파라미터 현황")
print(f"  총 파라미터:         {unet_total:,}개")
print(f"  인코더 (ResNet34):   {unet_enc:,}개  ← ImageNet 사전학습됨")
print(f"  디코더 + 출력 헤드:  {unet_total-unet_enc:,}개  ← 파인튜닝 대상")

In [ ]:
# ============================================================
# U-Net 패치 추론 데모
# ============================================================

PATCH = 256  # 패치 크기

def to_unet_tensor(data, patch=256):
    """
    (6,H,W) 배열에서 중앙 패치를 추출하고 U-Net 입력 텐서로 변환
    - DN → 반사율 (÷10000)
    - 채널별 정규화 (Sentinel-2 통계 기반)
    """
    h0 = (data.shape[1] - patch) // 2
    w0 = (data.shape[2] - patch) // 2
    p = data[:, h0:h0+patch, w0:w0+patch].astype(np.float32) / 10000.0
    # Sentinel-2 밴드 평균·표준편차 (글로벌 대략값)
    means = np.array([0.033, 0.044, 0.047, 0.200, 0.115, 0.072])[:,None,None]
    stds  = np.array([0.023, 0.028, 0.035, 0.082, 0.063, 0.052])[:,None,None]
    p = (p - means) / (stds + 1e-8)
    return torch.FloatTensor(p).unsqueeze(0), h0, w0  # (1,6,H,W)

post_t, h0, w0 = to_unet_tensor(post_data, PATCH)
pre_t,  _,  _  = to_unet_tensor(pre_data,  PATCH)

with torch.no_grad():
    unet_post = unet(post_t.to(device)).squeeze().cpu().numpy()
    unet_pre  = unet(pre_t.to(device)).squeeze().cpu().numpy()

print(f"U-Net 추론 완료")
print(f"  패치 크기: {PATCH}×{PATCH} 픽셀")
print(f"  출력 범위: {unet_post.min():.3f} ~ {unet_post.max():.3f}")
print()
print("참고: 이 모델은 산불 데이터로 파인튜닝되지 않았습니다.")
print("아키텍처와 추론 파이프라인 구조를 이해하는 것이 목표입니다.")
print("실제 탐지에는 labeled 데이터로 파인튜닝된 MLP 결과를 사용합니다.")

In [ ]:
# U-Net 인코더 특징 맵 시각화 (아키텍처 이해용)
feature_maps = {}

def make_hook(name):
    def hook(module, inp, out):
        feature_maps[name] = out.detach().cpu().numpy()
    return hook

hook = unet.encoder.layer1.register_forward_hook(make_hook('layer1'))
with torch.no_grad():
    _ = unet(post_t.to(device))
hook.remove()

layer1 = feature_maps['layer1'][0]  # (64, H/2, W/2)
print(f"Layer1 출력 형태: {layer1.shape}")
print("각 채널(필터)이 서로 다른 스펙트럴 패턴에 반응합니다.")

fig, axes = plt.subplots(4, 8, figsize=(16, 8))
fig.suptitle('ResNet34 인코더 Layer1 특징 맵 (32채널)\n'
             '각 필터가 다른 스펙트럴·텍스처 패턴을 감지함',
             fontsize=12)
for idx, ax in enumerate(axes.flat):
    if idx < 32:
        ax.imshow(layer1[idx], cmap='RdYlGn', aspect='auto')
    ax.axis('off')
plt.tight_layout()
plt.savefig('unet_feature_maps.png', bbox_inches='tight', dpi=150)
plt.show()

---
## 섹션 7. 이전·이후 비교 분석 및 통계

### 분석 방법 비교

| 방법 | 원리 | 장점 | 한계 |
|------|------|------|------|
| **dNBR 임계값** | 물리 기반 스펙트럴 규칙 | 해석 가능, 신뢰성 높음 | 임계값 수동 설정 |
| **MLP 딥러닝** | 다차원 스펙트럴 패턴 학습 | 복잡한 패턴 포착 | 학습 데이터 필요 |
| **U-Net** | 공간 맥락 고려 | 정밀한 경계선 | 대용량 레이블 필요 |

In [ ]:
# ============================================================
# 이후 영상 RGB 시각화용 NumPy 배열 준비
# ============================================================

def normalize_rgb(arr_rgb, valid_mask, pct=(2, 98)):
    """퍼센타일 기반 RGB 정규화 (0~1)"""
    out = np.zeros((*arr_rgb.shape[1:], 3), dtype=np.float32)
    for i, band in enumerate(arr_rgb):
        lo, hi = np.nanpercentile(band[valid_mask], pct)
        out[:, :, i] = np.clip((band - lo) / (hi - lo + 1e-8), 0, 1)
    out[~valid_mask] = 1.0  # nodata = 흰색
    return out

# 이후 영상: B4(R), B3(G), B2(B) 순서로 RGB 구성
post_rgb = normalize_rgb(post_data[[2, 1, 0]], valid)
pre_rgb  = normalize_rgb(pre_data[[2, 1, 0]],  valid)

# dNBR → 클래스 이미지
dnbr_cls = np.full((H, W), np.nan)
for i, (lo, hi) in enumerate(zip(DNBR_BINS[:-1], DNBR_BINS[1:])):
    mask_i = (dnbr_data[0] >= lo) & (dnbr_data[0] < hi) & valid
    dnbr_cls[mask_i] = i

print("시각화 배열 준비 완료")

In [ ]:
# ============================================================
# 최종 비교 시각화 (4-panel)
# ============================================================

cmap_sev = ListedColormap(SEV_COLORS)
norm_sev = BoundaryNorm(range(7), cmap_sev.N)

fig, axes = plt.subplots(1, 4, figsize=(20, 6))
fig.suptitle('산불 탐지 결과 비교\n2023 강릉·동해 산불 (이후 영상: 2023.04~05)',
             fontsize=13, fontweight='bold')

# 패널 1: 이전 진색합성
axes[0].imshow(pre_rgb)
axes[0].set_title('(1) 이전 진색합성\n(2023.02-03)', fontsize=10)
axes[0].axis('off')

# 패널 2: 이후 진색합성
axes[1].imshow(post_rgb)
axes[1].set_title('(2) 이후 진색합성\n(2023.04-05)', fontsize=10)
axes[1].axis('off')

# 패널 3: dNBR 심각도 분류
im3 = axes[2].imshow(dnbr_cls, cmap=cmap_sev, norm=norm_sev, interpolation='nearest')
axes[2].set_title('(3) dNBR 피해 심각도\n(물리 기반)', fontsize=10)
axes[2].axis('off')
plt.colorbar(im3, ax=axes[2], fraction=0.046, pad=0.04,
             ticks=range(1, 7), label='심각도 등급')

# 패널 4: MLP 확률 맵
im4 = axes[3].imshow(prob_map, cmap='RdYlGn_r', vmin=0, vmax=1, interpolation='bilinear')
axes[3].set_title('(4) MLP 딥러닝\n산불 피해 확률 (0~1)', fontsize=10)
axes[3].axis('off')
plt.colorbar(im4, ax=axes[3], fraction=0.046, pad=0.04, label='피해 확률')

plt.tight_layout()
plt.savefig('final_comparison.png', bbox_inches='tight', dpi=150)
plt.show()
print("'final_comparison.png' 저장 완료")

In [ ]:
# ============================================================
# NDVI 변화 통계 분석
# ============================================================

# 피해 / 정상 영역 마스크
burn_mask_np   = (dnbr_data[0] > BURN_THRESH)   & valid
normal_mask_np = (dnbr_data[0] < NORMAL_THRESH)  & (dnbr_data[0] > -0.1) & valid

# NDVI 추출
ndvi_pre_burn   = pre_idx[0][burn_mask_np]
ndvi_post_burn  = post_idx[0][burn_mask_np]
ndvi_pre_norm   = pre_idx[0][normal_mask_np]
ndvi_post_norm  = post_idx[0][normal_mask_np]

# 박스플롯 + 히스토그램
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('NDVI 변화 분석: 산불 피해 영역 vs. 정상 영역', fontsize=13)

# 박스플롯
data_box = [ndvi_pre_burn, ndvi_post_burn, ndvi_pre_norm, ndvi_post_norm]
labels_box = ['이전\n(피해지역)', '이후\n(피해지역)', '이전\n(정상지역)', '이후\n(정상지역)']
colors_box = ['#2ecc71', '#e74c3c', '#27ae60', '#c0392b']

bp = axes[0].boxplot(data_box, labels=labels_box, patch_artist=True, notch=True)
for patch, c in zip(bp['boxes'], colors_box):
    patch.set_facecolor(c); patch.set_alpha(0.7)
axes[0].axhline(0, color='gray', linestyle='--', alpha=0.5, label='NDVI=0')
axes[0].set_ylabel('NDVI 값')
axes[0].set_title('NDVI 분포 비교')
axes[0].legend(fontsize=9); axes[0].grid(True, alpha=0.3)

# 히스토그램
axes[1].hist(ndvi_pre_burn,  bins=60, alpha=0.6, color='#2ecc71',
             label=f'이전 (피해지역, n={len(ndvi_pre_burn):,})')
axes[1].hist(ndvi_post_burn, bins=60, alpha=0.6, color='#e74c3c',
             label=f'이후 (피해지역, n={len(ndvi_post_burn):,})')
axes[1].set_xlabel('NDVI 값'); axes[1].set_ylabel('픽셀 수')
axes[1].set_title('산불 피해 영역 NDVI 분포')
axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('ndvi_statistics.png', bbox_inches='tight', dpi=150)
plt.show()

# 통계 출력
print("\nNDVI 변화 통계 요약")
print("=" * 55)
print(f"{'지역':12} {'통계':6} {'이전':>8} {'이후':>8} {'변화':>9}")
print("-" * 55)
for region, pre_arr, post_arr in [
        ('산불 피해지', ndvi_pre_burn, ndvi_post_burn),
        ('정상 지역',  ndvi_pre_norm, ndvi_post_norm)]:
    for stat, pfn in [('평균', np.mean), ('중앙값', np.median)]:
        pv, qv = pfn(pre_arr), pfn(post_arr)
        chg = qv - pv
        print(f"{region:12} {stat:6} {pv:8.3f} {qv:8.3f} {chg:+9.3f}")
print("=" * 55)

In [ ]:
# ============================================================
# 피해 등급별 평균 NBR 변화 막대그래프
# ============================================================

sev_nbr_pre  = []
sev_nbr_post = []

for i in range(len(SEV_LABELS)):
    cls_mask = (dnbr_cls == i) & valid
    if cls_mask.sum() > 10:
        sev_nbr_pre.append(pre_idx[1][cls_mask].mean())
        sev_nbr_post.append(post_idx[1][cls_mask].mean())
    else:
        sev_nbr_pre.append(np.nan)
        sev_nbr_post.append(np.nan)

x = np.arange(len(SEV_LABELS))
w = 0.35

fig, ax = plt.subplots(figsize=(12, 5))
bars1 = ax.bar(x - w/2, sev_nbr_pre,  w, label='이전 NBR', color='#2ecc71', alpha=0.8)
bars2 = ax.bar(x + w/2, sev_nbr_post, w, label='이후 NBR', color='#e74c3c', alpha=0.8)

ax.set_xticks(x)
ax.set_xticklabels(SEV_LABELS, fontsize=10)
ax.set_ylabel('평균 NBR 값')
ax.set_title('피해 등급별 평균 NBR: 이전 vs. 이후\n(NBR 감소 클수록 심각한 산불 피해)')
ax.legend(fontsize=10)
ax.axhline(0, color='black', linewidth=0.8, linestyle='--')
ax.grid(axis='y', alpha=0.3)

for bar in bars1:
    h = bar.get_height()
    if not np.isnan(h):
        ax.text(bar.get_x() + bar.get_width()/2, h + 0.01,
                f'{h:.2f}', ha='center', va='bottom', fontsize=8)
for bar in bars2:
    h = bar.get_height()
    if not np.isnan(h):
        ax.text(bar.get_x() + bar.get_width()/2, h - 0.03,
                f'{h:.2f}', ha='center', va='top', fontsize=8, color='white')

plt.tight_layout()
plt.savefig('nbr_by_severity.png', bbox_inches='tight', dpi=150)
plt.show()

In [ ]:
# ============================================================
# 최종 요약 보고서
# ============================================================

print("=" * 65)
print("      2023 강릉·동해 산불 위성 분석 요약 보고서")
print("      Sentinel-2 기반 산불 피해 탐지 & 식생 변화")
print("=" * 65)

print(f"\n분석 지역   : 강원도 강릉·동해 일대")
print(f"ROI         : {128.85}~{129.25}E, {37.55}~{37.95}N")
print(f"분석 픽셀   : {valid.sum():,}개 (30m 해상도)")
print(f"분석 면적   : {valid.sum() * 900 / 1e6:.1f} km²")

print(f"\n분석 기간")
print(f"  이전: {PRE_START} ~ {PRE_END}")
print(f"  이후: {POST_START} ~ {POST_END}")

print(f"\n산불 피해 면적 추정")
print(f"  dNBR 기반 (경미 이상, >0.10): "
      f"{((dnbr_data[0]>0.10)&valid).sum()*900/10000:.0f} ha")
print(f"  dNBR 기반 (중간 이상, >0.27): {dnbr_burned_ha:.0f} ha")
print(f"  MLP 딥러닝 탐지:               {mlp_burned_ha:.0f} ha")

print(f"\n식생 변화 (NDVI, 산불 피해 영역)")
print(f"  이전 평균 NDVI: {ndvi_pre_burn.mean():.3f}")
print(f"  이후 평균 NDVI: {ndvi_post_burn.mean():.3f}")
change_pct = (ndvi_post_burn.mean()-ndvi_pre_burn.mean())/abs(ndvi_pre_burn.mean())*100
print(f"  변화량:         {ndvi_post_burn.mean()-ndvi_pre_burn.mean():+.3f} ({change_pct:.1f}%)")

print(f"\n딥러닝 모델")
print(f"  모델: BurnAreaMLP (PyTorch MLP, {total_params:,}개 파라미터)")
print(f"  학습: dNBR 의사 레이블 기반 반지도학습")
print(f"  최종 학습 정확도: {train_accs[-1]:.1f}%")

print("=" * 65)

---
## 섹션 8. 결론 및 응용

### 분석 결과 해석

1. **산불 피해 영역 탐지**: dNBR과 MLP 딥러닝 모두 유사한 피해 면적을 탐지합니다
2. **NDVI 감소**: 산불 피해 영역에서 NDVI가 크게 감소하여 식생 활력 손실이 명확히 나타남
3. **NBR 패턴**: 피해 심각도가 높을수록 NBR 감소량이 커지는 선형 관계 확인

### 실무 활용 방법

| 활용 분야 | 구체적 방법 |
|-----------|------------|
| **피해 면적 산정** | dNBR 심각도 분류 → 등급별 면적 계산 → 보험/복구 예산 산정 |
| **복원 우선순위** | 고심각도 구역 우선 복원 인력·예산 배분 |
| **회복 모니터링** | 매월 NDVI 추적 → 식생 회복 곡선 작성 (보통 2~5년) |
| **화재 위험 예측** | 건기 NDVI·토양수분·연료 수분 함량 모델링 |
| **탄소 배출 추정** | 피해 면적 × 바이오매스 밀도 × 연소율 계산 |

### 다음 단계 (심화 학습)

1. **더 많은 학습 데이터**
   - [EO4WildFires](https://huggingface.co/datasets/AUA-Informatics-Lab/eo4wildfires): 45개국, 31,730개 산불 이벤트
   - [CaBuAr](https://github.com/links-ads/igarss-burned-area): 캘리포니아 산불 이진 분할 데이터셋
   - [FLOGA](https://arxiv.org/abs/2311.03339): 그리스 산불, 전문가 레이블 포함

2. **U-Net 파인튜닝**: dNBR 레이블로 U-Net을 직접 학습하면 공간 정밀도 향상
3. **시계열 분석**: GEE 시계열 기능으로 산불 이후 식생 회복 모니터링 자동화
4. **SAR 데이터 융합**: Sentinel-1 SAR 데이터 추가 → 구름 관통 분석 가능
5. **Change Detection**: ChangeFormer, BitempNet 등 변화 탐지 특화 모델 탐색

---

### 참고 자료

- [UN-SPIDER Burn Severity Guide](https://un-spider.org/advisory-support/recommended-practices/burn-severity)
- [ESA Sentinel-2 User Guide](https://sentinels.copernicus.eu/web/sentinel/user-guides/sentinel-2-msi)
- [Google Earth Engine Documentation](https://developers.google.com/earth-engine)
- [geemap Documentation](https://geemap.org)
- [segmentation-models-pytorch](https://github.com/qubvel/segmentation_models.pytorch)

---

> **이 실습**은 오픈소스 데이터(Copernicus/Sentinel-2)와 무료 플랫폼(Google Earth Engine)만을 사용합니다.  
> 동일한 방법을 한국 내 모든 산불 사례에 적용할 수 있습니다.